# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [20]:
# imports

import os
import json
from dotenv import load_dotenv
import requests
from bs4 import BeautifulSoup
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [21]:
# constants

MODEL_GPT = 'gpt-4o-mini'


In [22]:
# Check OpenAi API Key

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")

# Use GPT model    
openai = OpenAI()


API key looks good so far


In [23]:
# Configure the system prompt for the helper agent
helper_system_prompt = """
You are an experienced pedriatrician assistant that analyzes the user's questions related to autistic children's health
and answers them thourougly but optimizing tokens output to keep answers concise. 
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
""" 

In [24]:
# User question

question_prompt = """
How can i help to improve my autistic child's sleep quality? and what are the common causes of sleep problems in children? 
Lasttley my child has been having nightmares, what can be the reason for that?
what book can i read to improve my knowledge about children's sleep?
"""

In [25]:
# Define the answer streamming function

def stream_answer(client, model, question):
    stream = client.chat.completions.create(
        model= model,
        messages=[
            {"role": "system", "content": helper_system_prompt},
            {"role": "user", "content": question_prompt}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [26]:
# Get gpt-4o-mini to answer, with streaming

stream_answer(openai, MODEL_GPT, question_prompt)

Improving your autistic child's sleep quality can involve several strategies:

### Strategies to Improve Sleep Quality
1. **Establish a Routine**: Implement a consistent bedtime routine to signal that it's time to wind down.
2. **Create a Comfortable Environment**: Ensure the bedroom is dark, quiet, and cool. Use blackout curtains or white noise machines if needed.
3. **Limit Screen Time**: Reduce exposure to screens at least an hour before bed, as blue light can interfere with melatonin production.
4. **Physical Activity**: Encourage physical activities during the day to help expend energy, but avoid vigorous exercise close to bedtime.
5. **Diet**: Avoid heavy meals or stimulants like caffeine before bed. Foods high in magnesium, like bananas, can promote relaxation.

### Common Causes of Sleep Problems
- **Sensory Sensitivities**: Unusual responses to noise, light, or textures might disrupt sleep.
- **Anxiety and Stress**: Changes in routine, school, or social situations can lead to increased anxiety at bedtime.
- **Medical Conditions**: Conditions like sleep apnea, allergies, or gastrointestinal issues can affect sleep.
- **Medications**: Some medications used to manage symptoms may impact sleep quality.

### Nightmares
Nightmares in autistic children can stem from:
- **Anxiety**: Daily stresses or difficulties can manifest as nightmares.
- **Changes**: Transitions or new environments may trigger anxiety-induced nightmares.
- **Media**: Exposure to scary content can influence dreams and lead to nightmares.

### Recommended Book
You might find *"The Sleep Solution: Why Your Sleep is Broken and How to Fix It"* by W. Chris Winter helpful. It offers insights into improving sleep for children and addresses various sleep-related issues.

By implementing routines and understanding the common causes of sleep disturbances, you can create a more restful environment for your child.

# Getting advices from WHO website on autism

In [27]:
#creating a web scrapper function to fetch website content and extract text and links

def fetch_website_content(url, parser="html.parser"):
    try:
        headers = {
            "User-Agent": "Mozilla/5.0"
        }
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, parser)

        return {
            "status": "success",
            "url": url,
            "title": soup.title.string if soup.title else None,
            "text": soup.get_text(separator=" ", strip=True),
            "links": [a.get("href") for a in soup.find_all("a", href=True)]
        }

    except requests.exceptions.RequestException as e:
        return {
            "status": "error",
            "url": url,
            "error": str(e)
        }

In [28]:
# fetching content from the WHO website about autism

autism_who = fetch_website_content("https://www.who.int/news-room/fact-sheets/detail/autism-spectrum-disorders")

if autism_who["status"] == "success":
    print( autism_who)
else:
    print("Error:", autism_who["error"])

{'status': 'success', 'url': 'https://www.who.int/news-room/fact-sheets/detail/autism-spectrum-disorders', 'title': '\r\n\tAutism\r\n', 'text': "Autism Skip to main content Global Regions WHO Regional websites Africa Americas South-East Asia Europe Eastern Mediterranean Western Pacific When autocomplete results are available use up and down arrows to review and enter to select. Select language Select language English العربية 中文 Français Русский Español Home Health Topics All topics A B C D E F G H I J K L M N O P Q R S T U V W X Y Z Resources Fact sheets Facts in pictures Multimedia Podcasts Publications Questions and answers Tools and toolkits Popular Dengue Endometriosis Excessive heat Herpes Mental disorders Mpox Countries All countries A B C D E F G H I J K L M N O P Q R S T U V W X Y Z Regions Africa Americas Europe Eastern Mediterranean South-East Asia Western Pacific WHO in countries Data by country Country presence Country cooperation strategies Country office profiles Strength

In [ ]:
def message_builder(website):
    if website["status"] == "success":
       content = [
           {
               "role": "system","content": helper_system_prompt
           },{
               "role": "user","content": question_prompt + website
           }
       ]
    else:
        content = "Sorry, I couldn't fetch the information from the WHO website."

    return content

In [30]:
#returning avice from LLM

message_builder(autism_who)

TypeError: can only concatenate str (not "dict") to str